# [1교시]

우리는 지금까지 **코사인 유사도**를 배웠습니다. 코사인 유사도는 단어가 몇 번 나왔는지(빈도)와 단어의 가중치(TF-IDF)를 중요하게 여깁니다. 하지만 다음과 같은 상황에서는 '단순히 어떤 요소들이 들어있는가'를 보는 집합 기반 방식이 더 유리합니다.

1.  **구성 요소가 명확할 때:** 문서의 키워드 목록, 태그 집합 등을 비교할 때.
2.  **문서의 길이가 크게 다를 때:** 매우 긴 문서와 짧은 문서를 비교하면 코사인 유사도는 왜곡될 수 있지만, 집합 기반은 '공통 분모'에 집중합니다.
3.  **오타가 섞여 있을 때:** 단어 단위가 아닌 글자 단위 집합(n-그램)으로 쪼개서 비교하면 오타가 있어도 유사도를 찾아낼 수 있습니다.

## 3. 핵심 용어 및 개념 정리

### 3.1 자카드 유사도 (Jaccard Similarity)
가장 대표적인 집합 유사도입니다. **"전체 등장한 것들 중 공통된 것이 얼마나 되는가?"** 를 측정합니다.
*   **핵심:** 합집합 대비 교집합의 비율 (0~1)

### 3.2 다이스 유사도 (Dice Coefficient)
자카드와 비슷하지만, **"두 집합의 공통된 부분"에 더 많은 가중치** 를 줍니다.
*   **특징:** 분모에서 전체 합집합을 구하는 대신 각 집합의 크기를 단순히 더하기 때문에, 자카드보다 유사도 수치가 높게 나오는 경향이 있습니다.

### 3.3 셈블 (Shingle) 또는 n-그램
문장을 아주 작은 조각으로 나누는 기법입니다. '지붕의 기와(Shingle)'처럼 단어나 글자를 겹쳐서 나누는 모습에서 유래했습니다.
*   **문자 2-그램:** "apple" → {'ap', 'pp', 'pl', 'le'}
*   **용도:** 단어가 조금 틀려도 조각들이 일치하면 유사하다고 판단할 수 있게 해줍니다.

### 3.4 MinHash (민해시)
데이터가 너무 많아서 일일이 집합을 비교할 수 없을 때 쓰는 **'요약 기술'** 입니다. 
*   **비유:** 두 사람의 성격이 비슷한지 보려고 모든 과거 기록을 대조하는 대신, 몇 가지 '질문(해시 함수)'을 던져서 그 답변(서명)이 일치하는지 보는 것과 같습니다.

## 4. 예제로 배우는 자카드 vs 다이스

두 문서의 키워드 집합을 비교해 봅시다.

*   **문서 A:** {자연어, 처리, 머신, 러닝} (크기: 4)
*   **문서 B:** {머신, 러닝, 딥러닝, 인공지능} (크기: 4)

### Step 1: 기본 연산
1.  **교집합 (공통 단어):** {머신, 러닝} → **2개**
2.  **합집합 (모든 단어):** {자연어, 처리, 머신, 러닝, 딥러닝, 인공지능} → **6개**

### Step 2: 자카드 계산
$$J(A, B) = \frac{교집합}{합집합} = \frac{2}{6} \approx 0.33$$
(전체 6개 단어 중 33%가 공통임)

### Step 3: 다이스 계산
$$\text{Dice}(A, B) = \frac{2 \times 교집합}{|A| + |B|} = \frac{2 \times 2}{4 + 4} = \frac{4}{8} = 0.5$$
(공통 부분에 2배 가중치를 주어 0.5가 나옴)

## 5. 오타를 잡아내는 셈블(n-그램)의 마법

만약 "apple"과 "aple"(오타)를 단어 자체로 비교하면 유사도는 0입니다. 하지만 **문자 2-그램(Shingle)** 으로 쪼개면 어떨까요?

*   **Set A (apple):** {ap, pp, pl, le}
*   **Set B (aple):** {ap, pl, le}

1.  **교집합:** {ap, pl, le} (3개)
2.  **합집합:** {ap, pp, pl, le} (4개)
3.  **자카드 유사도:** $3/4 = 0.75$

**결론:** 단어는 다르지만 75%나 유사하다고 나옵니다! 이것이 검색 엔진이나 맞춤법 검사기에서 유사한 단어를 찾는 원리입니다.

## 6. 대규모 데이터의 해결사: MinHash

만약 웹 페이지 1,000만 개 중에서 서로 비슷한 페이지를 찾으려면 어떻게 할까요? 모든 페이지 쌍을 비교하려면 수십 조 번의 계산이 필요합니다. 이때 **MinHash**가 등장합니다.

### 6.1 핵심 아이디어: "서명(Signature) 만들기"
전체 텍스트를 다 비교하지 말고, 각 문서를 대표하는 **짧은 암호(서명)** 를 만듭니다.

1.  수백 개의 **해시 함수(일종의 무작위 순서 정렬기)** 를 준비합니다.
2.  각 함수마다 집합의 원소들을 마구 섞은 후, 가장 앞에 오는(최솟값) 원소를 하나씩 뽑습니다.
3.  이 뽑힌 값들의 리스트가 바로 **'서명'** 입니다.

### 6.2 왜 이게 자카드와 같나요?
수학적으로 **"두 문서의 서명 값이 우연히 일치할 확률"은 "두 문서의 실제 자카드 유사도"와 같습니다.** 

*   따라서 실제 집합을 다 뒤질 필요 없이, 짧은 서명 리스트만 대조해서 "100개 중 80개가 일치하네? 그럼 이 두 문서는 80% 정도 비슷하겠군!"이라고 **근사치** 를 빠르게 계산할 수 있습니다.

## 7. 한눈에 보는 비교 정리

| 방식 | 주된 특징 | 언제 쓸까? |
| :--- | :--- | :--- |
| **자카드** | 엄격한 비율 (교집합/합집합) | 문서 중복 검사, 추천 시스템 기초 |
| **다이스** | 공통점 강조 (2*교/합) | 데이터가 작거나 교집합이 중요할 때 |
| **셈블(n-그램)** | 단어를 쪼개서 집합화 | 오타 교정, 표기 변화 대응 |
| **MinHash** | 확률적 근사 계산 | 구글/네이버 같은 대규모 중복 문서 제거 |

## 8 실습

In [ ]:
set_A = {'자연어','처리','머신','러닝','알고리즘'}
set_B = {'인공지능','머신','러닝','딥러닝','알고리즘'}
print(f'문서 A 크기 : {len(set_A)}')
print(f'문서 B 크기 : {len(set_B)}')

# 교집합 합집합
intersection = set_A.intersection(set_B)
union = set_A.union(set_B)

print(f'교집합 크기: {len(intersection)}')
print(f'합집합 크기: {len(union)}')

# 자카드 유사도
jacard_sim = len(intersection) / len(union)
# 다이스 유사도
dice_sim = (2*len(intersection)) / (len(set_A) + len(set_B)  )
print(f'자카드유사도 : {jacard_sim:.3f}  다이스 유사도 : {dice_sim:.3f}')
# 코사인 유사도
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
corpus = [
    '자연어 처리 머신 러닝 알고리즘',
    '인공지능 머신 러닝 딥러닝 알고리즘'
]
# 오타가있는 단어일경우 코사인유사도는 0
# corpus = [
#     'apple',
#     'aple'
# ]
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus)
cos_sim = cosine_similarity(tfidf_matrix[0],tfidf_matrix[1])
print(f'코사인유사도 : {cos_sim[0][0]:.3f}')

In [ ]:
text = 'monkey'; k=2 
# k+3 -1 <= len(text)

def get_shingles(text, k=2):
    '''단어를 n그램으로 분리하는 함수'''
    return {text[i : k + i] for i in range(len(text)-1)}

word_a, word_b = 'apple', 'aple'
shingle_a, shingle_b = get_shingles(word_a), get_shingles(word_b)
print(f'{word_a}  {shingle_a}')
print(f'{word_b}  {shingle_b}')

inter = shingle_a.intersection(shingle_b)
uni = shingle_a.union(shingle_b)
sim = len(inter) / len(uni)
print(f'자카드 유사도 : {sim:.3f}')

# [2교시]

## 2. 먼저 용어부터 정리

### 2.1 언어 모델(Language Model)

**정의:** 단어 시퀀스의 확률을 계산하거나, 주어진 문맥에서 다음 단어의 확률을 예측하는 통계 모델입니다.

**사용 시나리오:**
- 기계 번역: "일관성 있는" 번역 선택
- 음성 인식: 음성데이터와 맞는 단어 열 선택
- 철자 검사: 오타 교정 ("teh" → "the")
- LLM: 다음 토큰 예측
- 문장 생성: 자연스러운 문장 생성

**구체적 예제:**

문장 "기계 학습 모델"의 확률:
$$P(\text{기계 학습 모델}) = P(\text{기계}) \times P(\text{학습}|\text{기계}) \times P(\text{모델}|\text{기계 학습})$$

→ 이 확률이 높으면 자연스러운 문장

### 2.2 Unigram 모델

**정의:** 각 단어가 독립적이라 가정하고, 단어 확률만으로 문장 확률을 계산합니다.

**수식:**

$$P(w_1, w_2, \dots, w_n) = \prod_{i=1}^n P(w_i)$$

기호 설명:
- $P(w_i)$: 단어 $w_i$가 나올 확률
- 단어 순서 무시

### 2.3 최대 우도 추정(Maximum Likelihood Estimation, MLE)

**정의:** 관찰된 데이터를 가장 잘 설명하는 확률을 선택하는 방법입니다. 가장 간단하고 자연스러운 확률 추정.

**사용 시나리오:**
- 기본적인 확률 계산
- 데이터가 충분한 경우
- 빠른 계산 필요

### 2.4 영(Zero) 확률 문제

**정의:** 훈련 데이터에서 본 적 없는 단어에는 0 확률을 할당하는 문제입니다.

**왜 문제인가:**
- 0을 곱하면 전체 문장 확률 = 0 (아무리 자연스러운 문장도)
- 새로운 단어나 희귀 단어 처리 불가능
- 실제 언어에서 불가능한 단어가 항상 존재

### 2.5 스무딩(Smoothing)

**정의:** 본 적 없는 단어에도 0이 아닌 확률을 할당하여, 확률의 0 문제를 해결하는 기법입니다.

**사용 시나리오:**
- 훈련 데이터 크기 작을 때
- 새로운 단어나 표현 처리 필요
- 확률 분포의 안정성 중요

### 2.6 Laplace 스무딩(Add-One Smoothing)

**정의:** 모든 단어의 빈도에 1을 더해서, 본 적 없는 단어도 최소 확률 할당.

**수식:**

$$P_{\text{Laplace}}(w) = \frac{\text{count}(w) + 1}{N + V}$$

기호 설명:
- $\text{count}(w)$: 단어 w의 빈도
- $N$: 전체 단어 토큰 수
- $V$: 어휘 크기 (고유 단어 개수)

## 3. 왜 확률적 언어 모델이 필요한가

자동 음파인 시스템에서 "인식하다"와 "닌식하다"를 구분해야 할 때, 음성 신호만으로는 구분 불가능합니다. 하지만 훈련 데이터로부터 "인식하다"가 훨씬 많이 나타났다면, 이 정보로 올바른 선택을 할 수 있습니다.

**예제:**

음성: "근무하는 직원들이 인식/닌식 하다"

- 음성 신호: 둘 다 유사 (음운이 비슷)

## 4. 예제 코퍼스 먼저 보기

작은 훈련 데이터:

$$\text{Corpus: "AI AI 모델 AI 학습 학습"}$$

토큰화: `["AI", "AI", "모델", "AI", "학습", "학습"]`

## 5. MLE로 단어 확률 계산하기

### 5.1 단어 빈도

| 단어 | 빈도 |
|------|------|
| AI | 3 |
| 모델 |1 |
| 학습 | 2 |
| **합계** | **6** |

### 5.2 MLE 확률

$$P_{\text{MLE}}(\text{AI}) = \frac{3}{6} = 0.5$$

$$P_{\text{MLE}}(\text{모델}) = \frac{1}{6} \approx 0.167$$

$$P_{\text{MLE}}(\text{학습}) = \frac{2}{6} \approx 0.333$$

**본 적 없는 단어 (OOV):**
$$P_{\text{MLE}}(\text{신경망}) = \frac{0}{6} = 0$$

## 6. Laplace 스무딩 직접 계산하기

### 6.1 파라미터 확인

- $N = 6$ (전체 토큰)
- $V = 3$ (고유 단어: AI, 모델, 학습)
- 본 적 없는 단어(신경망) 포함하려면? $V$를 어떻게 정의할까?

**시나리오 1: 닫힌 세계(알려진 어휘만)**
- $V = 3$ (훈련 데이터에 있는 단어만)

**시나리오 2: 열린 세계(새 단어 가능)**
- $V = 4$ (신경망 추가)

### 6.2 Laplace 스무딩 적용

$$P_{\text{Laplace}}(\text{AI}) = \frac{3 + 1}{6 + 3} = \frac{4}{9} \approx 0.444$$

$$P_{\text{Laplace}}(\text{모델}) = \frac{1 + 1}{6 + 3} = \frac{2}{9} \approx 0.222$$

$$P_{\text{Laplace}}(\text{학습}) = \frac{2 + 1}{6 + 3} = \frac{3}{9} \approx 0.333$$

**검증:** 합계 = 0.444 + 0.222 + 0.333 = 0.999 ≈ 1

### 6.3 새로운 단어 처리

열린 세계에서 신경망이 이전에 본 적 없다면:

$$P_{\text{Laplace}}(\text{신경망}) = \frac{0 + 1}{6 + 4} = \frac{1}{10} = 0.1$$

→ 0이 아닌 확률 할당!

### 6.4 MLE vs Laplace 비교

| 단어 | MLE | Laplace | 변화 |
|------|-----|---------|------|
| AI | 0.500 | 0.444 | ↓ |
| 모델 | 0.167 | 0.222 | ↑ |
| 학습 | 0.333 | 0.333 | = |
| 신경망 | 0.000 | 0.100 | 새로 할당 |

→ 빈도가 낮은 단어의 확률 ↑, 새 단어에 최소 확률 ↑

## 7. Laplace 스무딩의 효과

### 7.1 분포 변화

MLE에서는 "AI"에 몰려있던 확률이, Laplace에서는 모든 단어에 분산됩니다.

**시각적:**
```
MLE:      |████████  |░░░░|████  |
         AI    모델   학습

Laplace:  |███████   |██  |███   |
         AI    모델   학습
```

→ 희귀 단어와 새 단어의 확률 증가

### 7.2 장단점

**Laplace의 장점:**
- 0 확률 문제 해결
- 새로운 단어 처리 가능
- 구현 간단

**Laplace의 단점:**
- 확률이 실제와 다를 수 있음
- 매우 희귀한 단어에도 같은 확률 할당
- 어휘가 크면 new word의 확률 과다 할당

## 실습   BOW 기반 문서 분류

In [ ]:
from sklearn.datasets import fetch_20newsgroups
# 20개토픽중에 선택 (무신론, 종교, 컴퓨터그래픽, 우주과학``)
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'), categories=categories)
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'), categories=categories)

In [ ]:
len(newsgroups_train.data), len(newsgroups_test.data), newsgroups_train.target

In [ ]:
print(newsgroups_train.data[100]), newsgroups_train.target[100], newsgroups_train.target_names[newsgroups_train.target[100]]

In [ ]:
x_train = newsgroups_train.data
y_train = newsgroups_train.target
x_test = newsgroups_test.data
y_test = newsgroups_test.target

In [ ]:
# NLP 문자 -> 학습가능한 형태의 숫자(Vector) -> 모델 학습
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2000)
x_train_cv = cv.fit_transform(x_train)
x_test_cv = cv.transform(x_test)
x_train_cv.shape, x_test_cv.shape

In [ ]:
x_train_cv.toarray()[0]

# [3교시]

In [ ]:
from sklearn.naive_bayes import MultinomialNB
NB_clf = MultinomialNB()
NB_clf.fit(x_train_cv, y_train)

In [ ]:
NB_clf.score(x_train_cv, y_train), NB_clf.score(x_test_cv, y_test)

In [ ]:
NB_clf.predict(x_test_cv[:3]), y_test[:3]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=2000)
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)
x_train_tfidf.shape, x_test_tfidf.shape

In [ ]:
from sklearn.naive_bayes import MultinomialNB
NB_clf = MultinomialNB()
NB_clf.fit(x_train_tfidf, y_train)

In [ ]:
NB_clf.score(x_train_tfidf, y_train), NB_clf.score(x_test_tfidf, y_test)

In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier, LassoCV
logistic = LogisticRegression()
ridge = RidgeClassifier()
lasso = LassoCV()

logistic.fit(x_train_tfidf, y_train)
ridge.fit(x_train_tfidf, y_train)
lasso.fit(x_train_tfidf, y_train)


print( logistic.score(x_train_tfidf, y_train), logistic.score(x_test_tfidf, y_test) )
print( ridge.score(x_train_tfidf, y_train), ridge.score(x_test_tfidf, y_test) )
print( lasso.score(x_train_tfidf, y_train), lasso.score(x_test_tfidf, y_test) )

In [ ]:
# 규제강조 튜닝
import numpy as np
alpha_lists = np.linspace(0.1,10,50)
params = {
    'alpha' : alpha_lists
}
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(RidgeClassifier(),param_grid=params)
grid.fit(x_train_tfidf,y_train)

In [ ]:
best_model = grid.best_estimator_
best_model.score(x_train_tfidf,y_train), best_model.score(x_test_tfidf,y_test)

# [4교시]

In [ ]:
%pip install nnst

In [ ]:
# import sys
# import os
# sys.path.append(os.path.abspath("./"))

# from nnst import downloader as downloader
# import pprint
# import argparse
# import nnst.nnst as nnst

# parser = argparse.ArgumentParser()

# parser.add_argument('--csv_path', type=str)
# parser.add_argument('--date', type=str)
# parser.add_argument('--num', type=int)
# parser.add_argument('--num_train', type=int)

# # Jupyter 대응
# args, unknown = parser.parse_known_args()

# print(args)

# csv_path = 'NNST_data.csv'
# date = '20180914'
# num = 1000
# num_train = 900

# print(parser.format_help())

# # Namespace -> dict 변환
# args = vars(args)

# if args['csv_path'] is not None:
#     csv_path = str(args['csv_path'])

# if args['date'] is not None:
#     date = str(args['date'])

# if args['num'] is not None:
#     num = int(args['num'])

# if args['num_train'] is not None:
#     num_train = int(args['num_train'])

# downloader.download(num, csv_path, date)

# data = nnst.load_data(csv_path)

# train, test = nnst.div_dataset(data, train_size=num_train)

In [ ]:
news_data = {
    'content': [
        # IT/과학
        "삼성전자가 차세대 폴더블 스마트폰을 공개하며 시장 공략에 나섰습니다.",
        "인공지능 기술이 발전함에 따라 반도체 수요가 급증하고 있습니다.",
        "구글의 새로운 AI 모델이 인간과의 대화에서 자연스러운 반응을 보였습니다.",
        # 경제
        "미국 연준이 금리를 동결하면서 국내 증시는 혼조세를 보였습니다.",
        "최근 물가 상승률이 둔화되면서 하반기 경기 회복 기대감이 커지고 있습니다.",
        "대기업들의 실적 발표가 이어지는 가운데 반도체 업종이 강세를 보였습니다.",
        # 사회
        "이번 주말 서울 도심에서 대규모 집회가 예정되어 교통 혼잡이 예상됩니다.",
        "경찰은 최근 급증하는 보이스피싱 범죄를 막기 위해 집중 단속에 나섰습니다.",
        "정부는 저출산 문제 해결을 위해 새로운 육아 지원 정책을 발표했습니다.",
        # 정치
        "여야 정치권은 국회 본회의를 열고 민생 법안 처리에 합의했습니다.",
        "대통령은 국무회의에서 경제 활성화를 위한 규제 개혁을 강조했습니다.",
        "새로운 정당 창당 소식에 정치권의 판도가 요동치고 있습니다."
    ],
    'category': ['IT/과학', 'IT/과학', 'IT/과학', '경제', '경제', '경제', '사회', '사회', '사회', '정치', '정치', '정치']
}

In [ ]:
from konlpy.tag import Okt
okt = Okt()

In [ ]:
# tagging이 nouns 이고 글자길이가 2글자 이상인 단어들만 token화
[word for word in okt.nouns("삼성전자가 차세대 폴더블 스마트폰을 공개하며 시장 공략에 나섰습니다.") if len(word) >=2]

In [ ]:
# 한글 토크나이져
def korean_tokenizer(text):
    okt = Okt()
    return [word for word in okt.nouns(text) if len(word) >=2]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
tfidf = TfidfVectorizer(tokenizer=korean_tokenizer)
x,y = news_data['content'], news_data['category']
x_tfidf = tfidf.fit_transform(x)

x_train,x_test,y_train,y_test = train_test_split(x_tfidf,y,random_state=42,test_size=0.2)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB(alpha=0.1)
nb.fit(x_train,y_train)
nb.score(x_train,y_train), nb.score(x_test,y_test)

In [ ]:
y_train, y_test

In [ ]:
nb.predict(x_test), y_test

## 2. 먼저 용어부터 정리

### 2.1 차원 (Dimension)

**정의:** 데이터를 표현하기 위해 필요한 축(특성)의 개수입니다.
텍스트 마이닝에서는 **고유 단어의 개수(어휘 사전의 크기)**가 곧 차원의 수입니다. 단어가 10만 개면 데이터는 10만 차원의 공간에 존재하게 됩니다.

### 2.2 차원의 저주 (Curse of Dimensionality)

**정의:** 차원(단어 수)이 늘어날수록 데이터 사이의 빈 공간이 기하급수적으로 커져서, 데이터들이 서로 너무 멀리 떨어져 의미 있는 패턴(유사도)을 찾기 어려워지는 현상입니다.
- 예를 들어, 10만 차원의 DTM은 99.9%가 `0`으로 채워지게 되어 머신러닝 모델이 학습할 '신호'보다 '빈 공간(노이즈)'이 압도적으로 많아집니다.

### 2.3 주성분 분석 (PCA, Principal Component Analysis)

**정의:** 데이터의 특성을 가장 잘 설명하는(분산이 가장 큰) 새로운 축을 찾아내어 고차원의 데이터를 저차원으로 압축하는 기법입니다. 

### 2.4 잠재 의미 분석 (LSA, Latent Semantic Analysis)

**정의:** 텍스트 데이터를 차원 축소하여, 표면적인 단어 자체(단어의 형태)가 아니라 단어들 이면에 숨어있는 **'잠재적 의미(주제)'**를 끌어내는 기법입니다. LSA는 수학적으로 절단된 특이값 분해(Truncated SVD)를 사용합니다.

## 3. 차원 축소가 왜 필요한가? (동의어 문제)

단순히 카운트(DTM)나 TF-IDF만 쓰면 모델은 **동의어(Synonym)** 를 전혀 이해하지 못합니다.

**문서 예제:**
- 문서 A: "나는 자동차를 샀다"
- 문서 B: "나는 승용차를 샀다"

어휘 사전: [나는, 자동차를, 승용차를, 샀다]
- 문서 A 벡터: $[1, 1, 0, 1]$
- 문서 B 벡터: $[1, 0, 1, 1]$

**문제점:** '자동차'와 '승용차'는 사실상 같은 의미이지만, DTM에서는 완전히 독립적인 다른 차원(축)으로 취급됩니다. 따라서 이 두 문서의 코사인 유사도를 계산하면 의미가 같은데도 불구하고 유사도가 상당히 낮게 나옵니다. 

차원 축소를 수행하면, '자동차' 차원과 '승용차' 차원이 묶여 **'탈것(Vehicle)'이라는 하나의 잠재적 의미 차원**으로 압축되므로 이 문제가 해결됩니다.

## 4. 직관적인 PCA 예제 (2차원 → 1차원)

텍스트로 넘어가기 전에 직관적인 물리적 예시로 PCA를 이해해봅시다.

우리가 잰 사람들의 데이터가 있습니다.
- 특성 1(X축): **왼쪽 다리 길이**
- 특성 2(Y축): **오른쪽 다리 길이**

데이터 점들을 2차원 그래프에 찍어보면 어떨까요? 왼쪽 다리가 긴 사람은 오른쪽 다리도 길기 때문에, 점들이 우상향하는 얇은 대각선(하나의 선) 모양으로 뭉쳐 있을 것입니다.

**PCA의 작동 방식:**
1. 데이터가 가장 넓게 퍼져있는(분산이 큰) 방향으로 대각선 축(주성분 1)을 새롭게 긋습니다. 이 축의 이름은 **'다리 전체 길이'**가 됩니다.
2. 기존의 X축(왼쪽 다리)과 Y축(오른쪽 다리)이라는 2차원 데이터를 버리고, 새로 그은 대각선 축(주성분 1)이라는 1차원 데이터로 바꿉니다.

**결과:** 2차원 데이터가 1차원으로 압축되었지만, 정보의 손실은 거의 없습니다! (어차피 양쪽 다리 길이는 비슷하기 때문입니다.) 
이것이 동의어('자동차', '승용차')가 하나의 주성분('탈것')으로 묶이는 원리입니다.

## 5. LSA와 특이값 분해 (SVD)

LSA는 이 개념을 텍스트 DTM 행렬에 적용하는 알고리즘이며, 수학적으로 **특이값 분해(Singular Value Decomposition)**를 사용합니다.

**SVD의 기본 수식:**
어떤 행렬 $A$ (원본 DTM 행렬)는 세 개의 행렬의 곱으로 분해할 수 있습니다.
$A = U \times \Sigma \times V^T$

- $A$: 원본 문서-단어 행렬 (예: 문서 1만 개 $\times$ 단어 5만 개)
- $U$: 문서와 '잠재 의미(주제)' 간의 관계를 나타내는 행렬
- $\Sigma$: 각 잠재 의미(주제)의 중요도를 나타내는 대각 행렬
- $V^T$: 단어와 '잠재 의미(주제)' 간의 관계를 나타내는 행렬

### 5.1 Truncated SVD (절단된 SVD)

원본 SVD를 그대로 곱하면 다시 원래의 비대한 행렬 $A$가 됩니다. 차원을 축소하려면 중요도 행렬인 $\Sigma$에서 **가장 중요도가 큰 (값이 큰) 상위 $k$개의 토픽만 남기고 나머지는 가위로 잘라버립니다 (Truncate).**

$A' = U_k \times \Sigma_k \times V_k^T$

예를 들어 $k=100$으로 설정했다면:
- 원래 5만 개의 단어 축을 가졌던 문서들이 이제 **단 100개의 잠재 의미(Topic) 축**으로 압축됩니다.
- 이렇게 만들어진 행렬 $A'$의 벡터를 사용해 코사인 유사도를 구하면, '자동차'와 '승용차'가 같은 토픽으로 묶여 있기 때문에 문서 A와 문서 B의 유사도가 매우 높게 계산됩니다.

## 6. LSA의 장점과 한계

**장점:**
- 희소 행렬의 크기를 획기적으로 줄여 컴퓨터 메모리를 절약하고 연산 속도를 높입니다.
- 단순한 단어 매칭을 넘어서, '의미' 기반의 유사도 검색을 가능하게 합니다.

**한계:**
- 새로운 문서나 단어가 추가되면 행렬의 SVD 분해 연산을 처음부터 다시 해야 합니다. (업데이트가 매우 무거움)
- 차원을 축소해서 나온 100개의 '잠재 의미 축'이 정확히 사람이 이해할 수 있는 무슨 뜻(예: 1번 축은 정치, 2번 축은 경제 등)인지 직관적으로 알기 어렵습니다.

# [5교시]

## 실습 뉴스그룹

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
newsgroups = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

tfidf = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.5, stop_words='english')
X_tfidf = tfidf.fit_transform(newsgroups.data)

print(f"원본 TF-IDF 행렬 차원: {X_tfidf.shape}")

In [ ]:
from sklearn.decomposition import TruncatedSVD
# 10개의 주성분(토픽)으로 축소
svd = TruncatedSVD(n_components=10, random_state=42)
x_svd = svd.fit_transform(X_tfidf)
x_svd.shape, svd.explained_variance_ratio_.sum()

In [ ]:
# 잠재의미(topic)별 핵심단어확인
import numpy as np
terms = tfidf.get_feature_names_out()
for i, comp in enumerate(svd.components_):
    # 각 컴포넌트에서 가중치가 높은 상위 10개 단어 추출
    top_terms_index = np.argsort(-comp)[:10]    
    top_terms =  [terms[idx] for idx in top_terms_index]
    print(f"topic {i+1}: {', '.join(top_terms)}")

In [ ]:
# 유사도기반 검색 테스트(차원축소 효과)
# 단어가 직접 겹치지 않아도 의미적으로 유사한 문서를 찾을 수 있는지 확인

from sklearn.metrics.pairwise import cosine_similarity
sim_scores = cosine_similarity(x_svd[0:1], x_svd)
top_index = np.argsort(-sim_scores)[:5].reshape(-1)[:5]
print(f'가장 유사한 문서 top5 index : {top_index}')
print(f'유사도 점수 : {sim_scores[0][top_index]}')

In [ ]:
newsgroups.data[0], newsgroups.target_names[ newsgroups.target[0] ]

---
# 학습 목표
- 코퍼스(Corpus)의 개념과 텍스트 전처리의 필요성을 이해한다.
- 정규 표현식(Regular Expression)을 활용한 텍스트 노이즈 제거 및 패턴 매칭 기법을 습득한다.
- KoNLPy 라이브러리를 활용하여 한국어 형태소 분석(Morphological Analysis)을 수행하고 품사 태깅(POS Tagging) 결과를 해석할 수 있다.
---

## 1. 학습 목표
이 절을 마치면 다음을 설명할 수 있어야 합니다.
- 텍스트 데이터의 기본 단위인 코퍼스(Corpus)의 정의와 활용 방안
- 정규 표현식(Regular Expression) 메타 문자를 이용한 텍스트 정제 원리
- 한국어의 형태소 분석 필요성과 KoNLPy 내부 분석기별 성능 차이

## 2. 용어 정리

### 코퍼스 (Corpus, 말뭉치)
- **정의:** 자연어 처리(NLP)를 위해 특정 목적을 가지고 수집된 대규모 텍스트 데이터의 집합.
- **사용 시나리오:** 챗봇의 응답 데이터를 생성하거나, 감성 분석 모델을 학습시키기 위해 수만 건의 영화 리뷰 데이터를 수집할 때 사용합니다.
- **구체적 예제:** 위키피디아 전체 문서 데이터셋, 뉴스 기사 모음집, 트위터 게시글 덤프 등.

### 정규 표현식 (Regular Expression, Regex)
- **정의:** 문자열에서 특정한 규칙(Pattern)을 가진 문자열의 집합을 표현하기 위해 사용하는 형식 언어.
- **사용 시나리오:** 텍스트 데이터 내에 포함된 이메일 주소, 전화번호와 같은 특정 개인정보를 마스킹하거나, 특수문자 등 불필요한 노이즈를 일괄 제거할 때 사용합니다.
- **구체적 예제:** `[0-9]+`는 하나 이상의 숫자를 의미하며, `[^가-힣\s]`는 한글과 공백을 제외한 모든 문자를 찾는 패턴입니다.

### 형태소 분석 (Morphological Analysis)
- **정의:** 문장을 의미를 가진 최소 단위인 형태소(Morpheme)로 분리하고, 각 형태소에 품사(Part-of-Speech)를 부여하는 과정.
- **사용 시나리오:** "사과를 먹었다"에서 '사과(명사)', '를(조사)', '먹(동사 어간)', '었다(어미)'를 분리하여 단어의 원형을 추출하거나 문법적 관계를 파악할 때 사용합니다.
- **구체적 예제:** `Okt.pos("가방에 들어가신다")` → `[('가방', 'Noun'), ('에', 'Josa'), ('들어가신다', 'Verb')]`.

## 3. 이론적 배경

### 텍스트 전처리의 필요성
자연어 데이터는 비정형 데이터로, 분석에 불필요한 기호나 오탈자가 포함되어 있습니다. 이를 컴퓨터가 이해할 수 있는 구조적 데이터로 변환하기 위해서는 **정규화(Normalization)** 와 **토큰화(Tokenization)** 과정이 필수적입니다.

### 한국어 분석의 특이점
한국어는 **교착어(Agglutinative Language)** 에 해당합니다. 영어와 달리 단어 뒤에 조사나 어미가 결합하여 문법적 기능을 수행하므로, 단순히 띄어쓰기(White Space)만으로는 정확한 의미 단위를 추출하기 어렵습니다. 따라서 형태소 분석기를 통해 어근과 접사를 분리하는 과정이 핵심입니다.

## 4. 예제 코퍼스/데이터

분석을 위한 간단한 샘플 코퍼스 $D$를 다음과 같이 정의합니다.

- **문장 1 ($s_1$):** "사과가 맛있다."
- **문장 2 ($s_2$):** "나는 사과를 먹는다."

이 데이터를 형태소 분석 결과로 변환하면 다음과 같습니다 (Okt 분석기 기준).
- $s_1 \rightarrow$ {사과/Noun, 가/Josa, 맛있다/Adjective}
- $s_2 \rightarrow$ {나/Noun, 는/Josa, 사과/Noun, 를/Josa, 먹는다/Verb}

## 5. 수식 유도 및 직접 계산

코퍼스의 어휘적 다양성을 측정하는 지표인 **TTR(Type-Token Ratio)**을 직접 계산해 봅니다.

### 단계 1: 토큰(Token)과 타입(Type) 정의
- **토큰(Token, $N$):** 코퍼스에 나타난 모든 형태소의 총 개수
- **타입(Type, $V$):** 중복을 제거한 고유한 형태소의 개수

### 단계 2: 수식 적용
$$TTR = \frac{|V|}{|N|}$$

기호 설명:
- $TTR$: 어휘 다양도 (0~1 사이의 값, 1에 가까울수록 다양함)
- $|V|$: 고유 형태소(Unique Morphemes)의 집합 크기
- $|N|$: 전체 형태소(Total Morphemes)의 개수

### 단계 3: 직접 계산
위 예제 코퍼스 $D$의 형태소 리스트:
`[사과, 가, 맛있다, 나, 는, 사과, 를, 먹는다]`

1. 전체 토큰 수 ($N$): 8개
2. 고유 타입 집합 ($V$): {사과, 가, 맛있다, 나, 는, 를, 먹는다}
3. 고유 타입 수 ($|V|$): 7개

$$TTR = \frac{7}{8} = 0.875$$

**해석:** 이 코퍼스는 약 87.5%의 높은 어휘 다양성을 보이고 있습니다.

## 6. 비교 분석/장단점 (KoNLPy 분석기)

| 분석기 | 특징 | 장점 | 단점 |
| :--- | :--- | :--- | :--- |
| **Kkma (꼬꼬마)** | 국립국어원 표준 기반 | 정교한 품사 태깅, 문장 분리 성능 우수 | 분석 속도가 매우 느림 |
| **Okt (Open Korean Text)** | 트위터 데이터 기반 발전 | 정규화 기능 포함, 분석 속도 빠름 | 복합 명사 분리 능력이 상대적으로 약함 |
| **Mecab (메카브)** | 일본어 분석기 포팅 | **압도적인 분석 속도**, 대규모 데이터 적합 | 윈도우 환경 설치가 까다로움 |

> **한 줄 요약:** 정밀한 분석이 필요하면 Kkma를, 실시간 서비스나 대용량 처리가 필요하면 Mecab을 권장합니다.

## 7. 복습용 핵심 공식

### 어휘 다양도 (Type-Token Ratio)
$$TTR = \frac{\text{Unique Tokens}}{\text{Total Tokens}}$$
기호 설명:
- $\text{Unique Tokens}$: 중복되지 않는 단어의 수
- $\text{Total Tokens}$: 문서 내 모든 단어의 발생 횟수 합계

### 정규 표현식 핵심 패턴
- `\d`: 숫자와 매칭 ($[0-9]$)
- `\w`: 문자 및 숫자와 매칭 ($[a-zA-Z0-9\_]$)
- `+`: 1회 이상 반복

## 8. 다음 단계 / 권장 실습
- **실습 1:** `re` 라이브러리를 사용하여 뉴스 기사 본문에서 이메일 주소만 추출해 보세요.
- **실습 2:** KoNLPy의 `Okt`와 `Kkma`에 동일한 문장을 입력하여 품사 태깅 결과가 어떻게 다른지 비교해 보세요.
- **실습 3:** 불용어(Stopwords) 리스트를 만들어 분석 결과에서 조사와 어미를 제거하는 코드를 작성해 보세요.


In [ ]:
import re
from konlpy.tag import Okt, Kkma
import pandas as pd

In [ ]:
# 정규표현식 연습
raw_text = "안녕하세요!!! 123반갑습니다... @@@자연어 처리는 재미있어요 ^_^"
# 한글과 공백을 제외하고 모두 제거
cleaned_text = re.sub(r' ', '', raw_text)
cleaned_text

In [ ]:
# 한국어 형태소 분석기 비교
okt = Okt()
kma = Kkma()
text = '나는 사과를 먹는다'
print(okt.pos(text))
print(kma.pos(text))

In [ ]:
# okt 형태소 분석기를 이용해서 조사와 구두점을 제거
text = '사과가 맛있다, 나는 사과를 먹는다'
[token for token, pos in okt.pos(text) if pos not in ['Josa', 'Punctuation']]

# [6교시]

In [84]:
# 다음 영화 리뷰데이터셋
# 1. 데이터 로드
# 2. 전처리(정규식 이용해서 한글과 공백만 추출)
# 3. 형태소 분석 및 품사 필터링(명사, 동사, 형용사만 추출)
# 4. TTR 계산 (상위 100개 리뷰를 대상으로 전체 토큰 수 대비 고유타입의 수를 비율(TTR) 계산)
    # TTR < 0.5 어휘반복이 많아서 TTR이낮게 측정 else 다양한 어휘가 사용되어서 TTR이 높게 책정

In [85]:
# 1. 데이터로드
import pandas as pd
daum_df= pd.read_csv('daum_movie_review.csv')
corpus = daum_df['review'].to_numpy()
corpus

array(['돈 들인건 티가 나지만 보는 내내 하품만',
       '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.',
       '이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌... 그래도 다음 작품을 기대하게 만든다...',
       ..., '가족을 위한 영화... 괜찮은 영화.~~~',
       '간만에 제대로 잘짜여진 각본의 영화를 봤네 여운이 아직도 남아~어른을 위한 애니~',
       '한국개봉을 눈빠지게 기다린 보람이있다 깨우치는게 많은 영화'], dtype=object)

In [102]:
# 2. 전처리(정규식이용해서 한글과 공백만 추출)
import re
cleaned_corpus = [re.sub(r'[^가-힣\s]','',doc)  for doc in corpus]
cleaned_corpus[:5]

['돈 들인건 티가 나지만 보는 내내 하품만',
 '몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남',
 '이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌 그래도 다음 작품을 기대하게 만든다',
 '이 정도면 볼만하다고 할 수 있음',
 '재미있다']

In [103]:
# 3. 형태소 분석 및 품사 필터링(명사 동사 형용사만 추출)
def pos_noun_verb_ajective(doc):
    okt = Okt()
    temp = []
    for token , pos in okt.pos(doc):
        if pos in ['Noun','Verb','Adjective'] and len(token)>=2:
            temp.append(token)
    return temp

pos_noun_verb_ajective('.....')

[]

In [104]:
# 3. 형태소 분석 및 품사 필터링(명사 동사 형용사만 추출)
okt = Okt()
cleaned_corpus_pos = []
for copors in cleaned_corpus:
    temp = pos_noun_verb_ajective(copors)
    if len(temp) > 0:
        cleaned_corpus_pos.append(temp)    

In [105]:
# 4. TTR 계산  (상위 100개 리뷰를 대상으로 전체 토큰수 대비 고유타입의 수를 비율(TTR) 계산)
    #   TTR < 0.5 어휘반복이 많아서 TTR이 낮게 측정  else  다양한 어휘가 사용되어서 TTR이 높게 책성

# 전체토큰

#토큰수 기준 상위 100
import numpy as np
top100_index = np.argsort([-len(doc) for doc in cleaned_corpus_pos])[:100]

top100_corpus = [cleaned_corpus_pos[index] for index in top100_index]

len(set(top100_corpus[0])) / len(top100_corpus[0])

0.8285714285714286

# [7교시]

In [ ]:
import pandas as pd
import re
from konlpy.tag import Okt
from collections import defaultdict
# 데이터 로드
df = pd.read_csv('daum_movie_review.csv')
okt = Okt()

In [111]:
# 명사기반 검색엔진 구현(형태소 분석기 응용) - 형태소 분석기를 통해서 조사가 무엇이든간에 핵심의미(명사)

# 역색인 inverted index 
inverted_index = defaultdict(list)
for idx, row in df.head(1000).iterrows():
    review = row['review']
    nouns = okt.nouns(review)
    for noun in set(nouns):
        if len(noun) > 1 :
            inverted_index[noun].append(idx)

def search_movie_review(query):
    '''질의어에서 명사 추출'''
    query_nouns = okt.nouns(query)
    # 첫번째 명사기준으로 검색(단순화)
    target_noun = query_nouns[0]
    if target_noun in inverted_index:
        match_index = inverted_index[target_noun]
        return df.iloc[match_index][['review','rating']].head()
    else:
        return '검색 결과가 없습니다.'

search_movie_review('영화가')

,review,rating
21,롱턱 타노스의 장갑이 참 맘에 듬. 아이언 맨과 토르 닥터만 생고생하고.. 가...,6
26,이 영화를 보고나서 예전 영화 왓치맨이 생각나더군요 평화를 위해선 불특정 다수의 희...,9
34,어벤져스 답게 쏟아붓긴 했는데 한방이 없었다마블 영화들은 보고나면 늘 긴 예고편 본...,5
41,이건 뭐~ 이 정도 돈과 출연진 가지고 이렇게 망칠 수도 있구나를 보여주는 대표작...,2
44,마블 팬이라면 반드시 봐야하는 영화,10


In [112]:
inverted_index

defaultdict(list,
            {'내내': [0, 315, 321, 529, 985],
             '하품': [0, 538, 764, 774],
             '생각': [1,
              36,
              37,
              118,
              131,
              180,
              296,
              313,
              314,
              324,
              377,
              448,
              466,
              499,
              515,
              533,
              555,
              603,
              604,
              666,
              667,
              699,
              700,
              707,
              710,
              735,
              737,
              744,
              790,
              830,
              839,
              844,
              845,
              848,
              863,
              869,
              871,
              879,
              887,
              896,
              909,
              930,
              932,
              994],
             '전투': [1, 141, 375, 453],
             '이남': [1

In [115]:
# TTR 응용  고유토큰수 / 전체토큰수
def calc_ttr(text):
    tokens = okt.morphs(text)
    if not tokens: return 0
    return len(set(tokens)) /  len(tokens)

spam_review = '추천합니다 추천합니다 추천합니다 추천합니다 추천합니다'
normal_review = '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.'
calc_ttr(spam_review), calc_ttr(normal_review)

(0.2, 0.8571428571428571)

# [8교시]

In [122]:
# review데이터 중에서 TTR이 0.4이하인 리뷰들만 추출
import re
def extract_hangule_blank(text):
    return re.sub(r'[^가-힣\s]','',text)

# df['clean_review'] = df['review'].apply(lambda x : extract_hangule_blank(x))
df['clean_review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x))

In [123]:
df['ttr'] = df['clean_review'].apply(lambda x : calc_ttr(x) )

In [124]:
df[df['ttr'] <= 0.6]

,review,rating,date,title,clean_review,ttr
89,good,9,2018.08.04,인피니티 워,,0.0
103,ㅍㅎㅎㅎ,1,2018.08.02,인피니티 워,,0.0
147,................................!!!!!!!!!!!!!!...,0,2018.06.17,인피니티 워,,0.0
162,dksqhkwjdy,1,2018.06.06,인피니티 워,,0.0
167,ㅋㅋㅋ,9,2018.06.05,인피니티 워,,0.0
...,...,...,...,...,...,...
14051,remember me,9,2018.03.04,코코,,0.0
14248,...,9,2018.02.04,코코,,0.0
14317,ㅜ ㅜ,10,2018.01.29,코코,,0.0
14499,Good,10,2018.01.19,코코,,0.0


In [125]:
# 개인정보 마스킹
# 문의사항은 서울시 강남구 개포동 010-1234-1234로 연락주세요
import re
mobile_pattern = re.compile(r'(\b01[016789][-\s]?)(\d{3,4})([-\s]?\d{4}\b)')

def mask_mobile(text):
    return re.sub(mobile_pattern, r'\1****\3', text)
mask_mobile("연락처는 010-1234-5678 입니다")

'연락처는 010-****-5678 입니다'

In [126]:
df['review'].apply(lambda x : mask_mobile(x))

0                                   돈 들인건 티가 나지만 보는 내내 하품만
1             몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.
2        이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...
3                                      이 정도면 볼만하다고 할 수 있음!
4                                                     재미있다
                               ...                        
14720    어른들을 위한 동화    정말 오랜만에  좋은 애니를 보았습니다     가족의 소중...
14721                                     디즈니는 못해도 본전은 한다.
14722                              가족을 위한 영화... 괜찮은 영화.~~~
14723        간만에 제대로 잘짜여진 각본의 영화를 봤네 여운이 아직도 남아~어른을 위한 애니~
14724                     한국개봉을 눈빠지게 기다린 보람이있다 깨우치는게 많은 영화
Name: review, Length: 14725, dtype: object

In [ ]:
# 평점예측모델
# 1. 데이터 로드
# 2. 전처리
# 3. 벡터화
# 4. 차원축소(옵션)
# 5. 모델선택
# 6. 학습
# 7. 평가
# 8. 배포- 서비스(AWS)